# **Rolling Stock ETL**

### Data Fetching

In [1]:
import pandas as pd
import psycopg2

def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    """
    Connects to PostgreSQL and loads the given table into a Pandas DataFrame.
    """
    try:
        # Connect to PostgreSQL
        connection = psycopg2.connect(
            host=host_ip,
            database=database_name,
            user=user,
            password=password,
            port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")

        # Create query
        query = f"SELECT * FROM {table_name};"

        # Load into pandas DataFrame
        df = pd.read_sql_query(query, connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")

        return df

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

    finally:
        if connection:
            connection.close()

# --- Configuration (same as before) ---
HOST_IP = "100.95.110.69"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

Connected successfully to pradigma-extractor on 100.95.110.69


C:\Users\win 11\AppData\Local\Temp\ipykernel_22588\3393819530.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, connection)


✅ Fetched 6834 rows from 'extraction'


In [2]:
df = df_original.copy(deep=True)
df = df[(df['status_id'] == 1) & (df['dept_name'] == 'Rolling-Stock')][['filename', 'workorder_id', 'json_data']]

df.head(3)

,filename,workorder_id,json_data
0,RS_PM_WEK_4000586856.pdf,4.000587e+09,{'notification': {'notification_no': '12244350...
1,RS_PM_MTH_4000464193.pdf,4.000464e+09,{'notification': {'notification_no': '11945789...
3,RS_PM_MTH_4000446287.pdf,4.000446e+09,{'notification': {'notification_no': '11898977...


In [3]:
len(df)

1206

In [4]:
import pandas as pd

valid_json = df['json_data']
valid_json = valid_json[valid_json.apply(lambda x: isinstance(x, dict))]

all_keys = set()
for item in valid_json:
    all_keys.update(item.keys())

print(sorted(all_keys))

['air_standup', 'airbag-pressure', 'airbag_pressure', 'approval', 'cardan_shaft', 'cceb', 'greasing_cardan_shaft', 'notification', 'stamping', 'technician', 'train_startup_test', 'tyre-pressure', 'tyre-wear', 'tyre_pressure', 'tyre_wear', 'water_ponding', 'work_order']


In [5]:
rename_map = {
    'airbag-pressure': 'airbag_pressure',
}

df = df.copy(deep=True)
df['json_data'] = df['json_data'].apply(
    lambda data: {rename_map.get(k, k): v for k, v in data.items()} if isinstance(data, dict) else data
)


### Airbag Pressure

In [6]:
df['airbag_pressure'] = df['json_data'].apply(
    lambda x: x.get('airbag_pressure') if isinstance(x, dict) else None
)

df['workorder_id'] = (
    df['workorder_id']
    .apply(lambda x: int(x) if pd.notnull(x) else None)
)

df[['filename', 'workorder_id', 'airbag_pressure']].head(1).to_dict(orient='records')

[{'filename': 'RS_PM_WEK_4000586856.pdf',
  'workorder_id': 4000586856,
  'airbag_pressure': {'bogie1': {'bogie_sn': 'null',
    'pressure_before': '5',
    'pressure_after': '5',
    'height_before': '241',
    'height_after': '241'},
   'bogie2': {'bogie_sn': 'null',
    'pressure_before': '5',
    'pressure_after': '5',
    'height_before': '241',
    'height_after': '241'},
   'bogie3': {'bogie_sn': 'null',
    'pressure_before': '4.5',
    'pressure_after': '4.5',
    'height_before': '241',
    'height_after': '241'},
   'bogie4': {'bogie_sn': 'null',
    'pressure_before': '4.5',
    'pressure_after': '4.5',
    'height_before': '242',
    'height_after': '242'},
   'bogie5': {'bogie_sn': 'null',
    'pressure_before': '4.5',
    'pressure_after': '4.5',
    'height_before': '242',
    'height_after': '242'},
   'bogie6': {'bogie_sn': 'null',
    'pressure_before': '4.5',
    'pressure_after': '4.5',
    'height_before': '241',
    'height_after': '241'},
   'bogie7': {'bogie_sn

In [7]:
import pandas as pd
import numpy as np

na_like_values = ['NA', 'N/A', 'NULL', 'NONE', 'NAN']

def is_na_like(val):
    if isinstance(val, (list, dict, np.ndarray)):
        return False
    try:
        if pd.isna(val):
            return True
    except Exception:
        pass
    val_str = str(val).strip().upper()
    return val_str in na_like_values

def find_na_keys(d):
    if not isinstance(d, dict):
        return []
    return [k for k, v in d.items() if is_na_like(v)]

df['na_keys'] = df['airbag_pressure'].apply(find_na_keys)

df_with_na = df[df['na_keys'].apply(lambda x: len(x) > 0)]

df_with_na[['filename', 'workorder_id', 'na_keys']].head(5)

,filename,workorder_id,na_keys


In [8]:
from collections import Counter

na_counter = Counter(k for keys in df['na_keys'] for k in keys)
na_summary = pd.DataFrame(na_counter.items(), columns=['key', 'na_count']).sort_values('na_count', ascending=False)

print(na_summary)

Empty DataFrame
Columns: [key, na_count]
Index: []


In [9]:

import numpy as np
import re
import pandas as pd

pattern = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

def clean_value(val):
    """Clean individual values (string, dict, etc.)."""
    if isinstance(val, str):
        return '' if pattern.match(val) else val
    elif isinstance(val, dict):
        return {k: clean_value(v) for k, v in val.items()}
    elif isinstance(val, list):
        return [clean_value(v) for v in val]
    else:
        return '' if pd.isna(val) else val

df['airbag_pressure'] = df['airbag_pressure'].apply(clean_value)

df['airbag_pressure'] = df['airbag_pressure'].replace(np.nan, '', regex=True)

df['airbag_pressure'].head(3)

0    {'bogie1': {'bogie_sn': '', 'pressure_before':...
1    {'bogie1': {'bogie_sn': '231', 'pressure_befor...
3    {'bogie1': {'bogie_sn': '257', 'pressure_befor...
Name: airbag_pressure, dtype: object

In [10]:
def extract_leaf_keys(d, parent=''):
    keys = []
    if isinstance(d, dict):
        for k, v in d.items():
            full_key = f"{parent}.{k}" if parent else k
            if isinstance(v, dict):
                keys.extend(extract_leaf_keys(v, full_key))
            else:
                keys.append(full_key)
    return keys

# Extract leaf keys for each row
df['airbag_pressure_leaf_keys'] = df['airbag_pressure'].apply(
    lambda x: extract_leaf_keys(x) if isinstance(x, dict) else []
)

# Combine and deduplicate all keys across the entire DataFrame
unique_keys = sorted(set(k for sublist in df['airbag_pressure_leaf_keys'] for k in sublist))

# Print all unique keys
for k in unique_keys:
    print(k)


approval.date
approval.supervisor_id
approval.technician_id
bogie1.bogie_sn
bogie1.height_after
bogie1.height_before
bogie1.pressure_after
bogie1.pressure_before
bogie2.bogie_sn
bogie2.height_after
bogie2.height_before
bogie2.pressure_after
bogie2.pressure_before
bogie3.bogie_sn
bogie3.height_after
bogie3.height_before
bogie3.pressure_after
bogie3.pressure_before
bogie4.bogie_sn
bogie4.height_after
bogie4.height_before
bogie4.pressure_after
bogie4.pressure_before
bogie5.bogie_sn
bogie5.height_after
bogie5.height_before
bogie5.pressure_after
bogie5.pressure_before
bogie6.bogie_sn
bogie6.height_after
bogie6.height_before
bogie6.pressure_after
bogie6.pressure_before
bogie7.bogie_sn
bogie7.height_after
bogie7.height_before
bogie7.pressure_after
bogie7.pressure_before
bogie8.bogie_sn
bogie8.height_after
bogie8.height_before
bogie8.pressure_after
bogie8.pressure_before


In [11]:
import json
import pandas as pd

def flatten_with_descriptions(row):
    flat = {}

    def recurse(subdict, parent=''):
        # Skip if None or not a dict or string
        if subdict is None:
            return

        # If it's a JSON string, parse it
        if isinstance(subdict, str):
            try:
                subdict = json.loads(subdict)
            except json.JSONDecodeError:
                return

        # If still not a dict, skip
        if not isinstance(subdict, dict):
            return

        for k, v in subdict.items():
            if len(k) == 1 and k.isalpha():
                new_parent = parent
            else:
                new_parent = f"{parent}.{k}" if parent else k

            if isinstance(v, dict):
                desc = v.get('description')
                if desc:
                    desc_key = (
                        desc.lower()
                        .replace(' ', '_')
                        .replace('/', '_')
                        .replace('&', 'and')
                    )
                    for sub_k, sub_v in v.items():
                        if sub_k != 'description':
                            flat[f"{new_parent}.{desc_key}.{sub_k}"] = sub_v
                else:
                    recurse(v, new_parent)
            else:
                flat[new_parent] = v

    recurse(row)
    return flat

# Apply flattening safely
flattened_rows = [flatten_with_descriptions(r) for r in df['airbag_pressure'].fillna({})]

# Create DataFrame
airbagpressure_df = pd.DataFrame(flattened_rows)
airbagpressure_df.index = df.index
airbagpressure_df['workorder_id'] = df['workorder_id'].astype('Int64')
airbagpressure_df['filename'] = df['filename']

airbagpressure_df


,bogie1.bogie_sn,bogie1.pressure_before,bogie1.pressure_after,bogie1.height_before,bogie1.height_after,bogie2.bogie_sn,bogie2.pressure_before,bogie2.pressure_after,bogie2.height_before,bogie2.height_after,...,bogie8.bogie_sn,bogie8.pressure_before,bogie8.pressure_after,bogie8.height_before,bogie8.height_after,approval.date,approval.technician_id,approval.supervisor_id,workorder_id,filename
0,,5,5,241,241,,5,5,241,241,...,,5,5,241,241,25/02/2024,11515,7127,4000586856,RS_PM_WEK_4000586856.pdf
1,231,5.1,,243,,213,4.4,,240,,...,219,5.1,,240,,12/05/2022,7306,7066,4000464193,RS_PM_MTH_4000464193.pdf
3,257,4.8,4.8,243,243,258,4.8,4.8,241,241,...,264,4.8,4.8,242,242,25/01/2022,7205,7127,4000446287,RS_PM_MTH_4000446287.pdf
4,244,5.0,,241,,228,4.8,,241,,...,246,5,,241,,09/10/2023,7205,7192,4000558454,RS_PM_WEK_4000558454.pdf
5,257,4.9,,241,,258,4.8,,241,,...,264,4.9,,241,,17/05/2022,7205,7127,4000464732,RS_PM_MTH_4000464732.pdf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6714,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4000614023,RS_PM_QTR_4000614023.pdf
6715,227,5.0,5.0,241,241,262,4.9,4.9,241,241,...,246,5.0,5.0,241,241,14/12/2022,11517,7196,4000501707,RS_PM_MTH_4000501707.pdf
6717,244,5.0,5.0,241,241,228,4.8,4.8,241,241,...,246,5.0,5.0,241,241,30/05/2023,11517,7196,4000522314,RS_PM_WEK_4000522314.pdf
6759,219,4.8,4.8,241,241,282,4.8,4.8,241,241,...,208,5.2,5.2,242,242,08/09/2024,7205,7192,4000625781,RS_PM_WEK_4000625781.pdf


In [12]:
for i, col in enumerate(airbagpressure_df.columns, start=1):
    print(f"{i:3d}. {col}")

  1. bogie1.bogie_sn
  2. bogie1.pressure_before
  3. bogie1.pressure_after
  4. bogie1.height_before
  5. bogie1.height_after
  6. bogie2.bogie_sn
  7. bogie2.pressure_before
  8. bogie2.pressure_after
  9. bogie2.height_before
 10. bogie2.height_after
 11. bogie3.bogie_sn
 12. bogie3.pressure_before
 13. bogie3.pressure_after
 14. bogie3.height_before
 15. bogie3.height_after
 16. bogie4.bogie_sn
 17. bogie4.pressure_before
 18. bogie4.pressure_after
 19. bogie4.height_before
 20. bogie4.height_after
 21. bogie5.bogie_sn
 22. bogie5.pressure_before
 23. bogie5.pressure_after
 24. bogie5.height_before
 25. bogie5.height_after
 26. bogie6.bogie_sn
 27. bogie6.pressure_before
 28. bogie6.pressure_after
 29. bogie6.height_before
 30. bogie6.height_after
 31. bogie7.bogie_sn
 32. bogie7.pressure_before
 33. bogie7.pressure_after
 34. bogie7.height_before
 35. bogie7.height_after
 36. bogie8.bogie_sn
 37. bogie8.pressure_before
 38. bogie8.pressure_after
 39. bogie8.height_before
 40. bogi

In [13]:
for i, col in enumerate(airbagpressure_df.columns, start=1):
    print(f"{i:3d}. {col}")

    if col == 'workorder_id':
        continue

    valid_workorders = airbagpressure_df.loc[airbagpressure_df[col].notna(), 'workorder_id'].unique()

    if len(valid_workorders) > 0:
        workorder_list = ", ".join(map(str, valid_workorders))
        print(f"   Work Orders with data ({len(valid_workorders)}): {workorder_list}")
        print("-" * 80)


  1. bogie1.bogie_sn
   Work Orders with data (1202): 4000586856, 4000464193, 4000446287, 4000558454, 4000464732, 4000457924, 4000449255, 4000493210, 4000453153, 4000453431, 4000460878, 4000449251, 4000487566, 4000460876, 4000616771, 4000443565, 4000458057, 4000457726, 4000629069, 4000446285, 4000290580, 4000445410, 4000443515, 4000449254, 4000453217, 4000561801, 4000548185, 4000462536, 4000629779, 4000554182, 4000621978, 4000554057, 4000492477, 4000680359, 4000462534, 4000497784, 4000496417, 4000490238, 4000464730, 4000467264, 4000476338, 4000475575, 4000476339, 4000475576, 4000489152, 4000511157, 4000473611, 4000470656, 4000479375, 4000469186, 4000467263, 4000473835, 4000470659, 4000509731, 4000506095, 4000505664, 4000503974, 4000493840, 4000478753, 4000523943, 4000511160, 4000518913, 4000480744, 4000511496, 4000501704, 4000495936, 4000519624, 4000553939, 4000584992, 4000524239, 4000535698, 4000538228, 4000513835, 4000539394, 4000550832, 4000538230, 4000544001, 4000541671, 4000541633

In [14]:
output_path = '../../output/rolling_stock.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    airbagpressure_df.to_excel(writer, index=False, sheet_name='airbag_pressure')

print(f"✅ Exported successfully to '{output_path}' (replaced existing sheet)")

✅ Exported successfully to '../../output/rolling_stock.xlsx' (replaced existing sheet)
